# 给 Agent 加记忆：MemGPT、Cartridges 与 CacheBlend

> 上一讲我们把 Agent 放进长代码库与长 issue 里反复工作，上下文窗口的边界被直接撞到：信息太多，窗口装不下。这一讲正面处理这条边界，讨论怎样让一个窗口有限的 Agent 拥有接近无限的记忆。
>
> 我们从三条路线分别作答：MemGPT 把上下文当作稀缺内存，让模型自己决定换入换出；Cartridges 把整份语料压缩进可训练表示；CacheBlend 在工程层复用预计算的 KV 缓存。三条路线合起来，是本讲对记忆的完整回答。


先给一个具体场景。一个客服助手三个月前接待过一位用户，记住了他的称呼、住址与沟通偏好。三个月后用户再次出现，助手若能记起这些，开场就能叫对名字、省去重复核对；若完全忘了，用户要再填一遍表单。区别不在模型参数，而在助手有没有把信息存下来、能不能在需要时取回。

本讲的起点是一组事实：LLM 的上下文窗口是一个硬上限；窗口内不同位置的记忆强度并不均匀，两头的 token 比中间更容易被注意到；任务需要的相关信息却常常落在窗口之外。要让有限窗口承载无限信息，只能靠更聪明的存储与调度，这正是记忆系统要解决的问题。我们从最直观的窗口预算开始，先算出对话会在第几轮撑爆窗口。


## 1. 记忆为什么重要

上下文窗口给每个 Agent 划了一条硬边界。模型推理时只能看到窗口内的 token，窗口外的一切都不存在。我们给各个组成部分分配一个量级，就能估算一段对话能撑多少轮：

| 组成部分 | 每轮量级（token） |
|:---|:---|
| 系统指令（固定） | 约 1000 |
| 用户消息 | 约 200 |
| 助手回复 | 约 300 |
| 检索结果 | 约 200 |

以 8k 窗口为例，扣掉固定开销后留给对话的约 7000 token，每轮消耗约 700 token，十轮左右窗口就被占满。窗口扩大到 128k 也只是把这一刻推迟，不能消除它。

窗口内部的记忆强度也不均匀。Lost in the Middle 的实证表明，模型对上下文两头的 token 记得更好，中间部分容易被忽略。把窗口塞满并不等于记住全部信息。要突破窗口的限制，只能改变信息的组织方式：逐出、压缩、复用。先从窗口预算的数值算起。


In [ ]:
# 手算：一个 8k 窗口的 Agent 能撑多少轮对话
window = 8000
system = 1000               # 系统指令固定占用
per_turn = 700              # 用户 200 + 助手 300 + 检索结果 200
usable = window - system
turns = usable // per_turn
warn = int(0.7 * window)
warn_turn = (warn - system) // per_turn + 1
print(f"可用对话预算 = {window} - {system} = {usable} token")
print(f"每轮消耗 {per_turn} token，约 {turns} 轮后占满窗口")
print(f"70% 警告线 {warn} token 出现在第 {warn_turn} 轮附近")
print(f"关键观察：{turns} 轮就撑爆窗口，而长期任务远超这个轮数")


In [ ]:
# 可视化：主上下文占用随轮次线性增长
import matplotlib.pyplot as plt
import numpy as np

window = 8000
warn = int(0.7 * window)
per_turn = 700
system = 1000
turn_ids = np.arange(0, 16)
occupancy = system + per_turn * turn_ids

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(turn_ids, occupancy, marker="o", label="main context")
ax.axhline(window, color="r", linestyle="--", label="window limit (100%)")
ax.axhline(warn, color="orange", linestyle="--", label="warning line (70%)")
ax.set_xlabel("turn")
ax.set_ylabel("tokens")
ax.set_title("Context occupancy vs dialogue turns")
ax.legend()
plt.tight_layout()
plt.show()

first_warn = np.flatnonzero(occupancy >= warn)[0]
first_full = np.flatnonzero(occupancy >= window)[0]
print(f"第 {first_warn} 轮越过警告线，第 {first_full} 轮到达上限")


信息本身不会消失，但主上下文必须保持有界。逐出（eviction）决定哪条消息让位：FIFO 按到达顺序逐出最老的；重要性打分按语义价值逐出最不重要的；LRU 按最近使用时间逐出最久没被用到的。三种策略对同一批消息会选出不同的让位者。我们用一段玩具数据比较它们。


In [ ]:
# 三条待管理记忆：到达顺序、重要性、最近使用时间
memories = [
    {"id": "a", "ts": 0, "importance": 0.3, "last_used": 5},
    {"id": "b", "ts": 1, "importance": 0.9, "last_used": 8},
    {"id": "c", "ts": 2, "importance": 0.6, "last_used": 2},
    {"id": "d", "ts": 3, "importance": 0.2, "last_used": 0},
]


def evict_fifo(memories):
    """FIFO：逐出到达最早（ts 最小）的一条，返回其下标。"""
    return min(range(len(memories)), key=lambda i: memories[i]["ts"])


def evict_lowest_importance(memories):
    """重要性：逐出打分最低的一条，返回其下标。"""
    return min(range(len(memories)), key=lambda i: memories[i]["importance"])


def evict_lru(memories):
    """LRU：逐出最近使用时间最早的一条，返回其下标。"""
    return min(range(len(memories)), key=lambda i: memories[i]["last_used"])


for name, fn in [("FIFO", evict_fifo),
                 ("importance", evict_lowest_importance),
                 ("LRU", evict_lru)]:
    idx = fn(memories)
    print(f"{name:10s} 逐出 {memories[idx]['id']}")
print("关键观察：同一批记忆，三种策略选出的让位者不同")


In [ ]:
# 可视化：三种策略下记忆内容随轮次变化
import matplotlib.pyplot as plt
import numpy as np


def simulate(evict_fn, seq, capacity):
    """依次放入 seq，超容量时按 evict_fn 逐出，返回每轮保留的 id 集合。"""
    kept = []
    store = []
    for item in seq:
        store.append(item)
        while len(store) > capacity:
            store.pop(evict_fn(store))
        kept.append(sorted(m["id"] for m in store))
    return kept


rng = np.random.default_rng(0)
seq = [{"id": f"m{i}", "ts": i,
        "importance": round(rng.uniform(0.1, 1), 2),
        "last_used": int(rng.integers(0, 12))} for i in range(12)]
capacity = 4

policies = {"FIFO": evict_fifo,
            "importance": evict_lowest_importance,
            "LRU": evict_lru}
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, fn) in zip(axes, policies.items()):
    kept = simulate(fn, seq, capacity)
    grid = np.zeros((len(seq), len(kept)))
    for t, ids in enumerate(kept):
        for item_id in ids:
            grid[int(item_id[1:]), t] = 1
    ax.imshow(grid, aspect="auto", cmap="Blues", interpolation="nearest")
    ax.set_title(name)
    ax.set_xlabel("turn")
    ax.set_ylabel("memory id")
plt.tight_layout()
plt.show()

print("每格代表一条记忆在某轮是否仍在主上下文；三种策略的保留形态不同。")


## 2. MemGPT：把上下文当作物理内存

操作系统在物理内存之外加一块磁盘，用分页机制把暂时用不到的数据换出、需要时再换回，从而在有限的物理内存上运行远比内存大的程序。MemGPT 把同一套思路搬给 LLM：上下文窗口相当于物理内存，外部存储相当于磁盘，函数调用相当于换页指令。

MemGPT 把主上下文分成三段固定顺序的部分。system instructions 只读，写明控制流与函数用法；working context 是一段可读写的定长文本，专门放用户关键事实与偏好；FIFO queue 是滚动消息历史，存最近几轮对话。主上下文之外是外部上下文，包含两个库：recall storage 存全部历史消息（永不删除），archival storage 存需要长期保存的档案。信息从主上下文换出后进入外部上下文，需要时由模型发函数调用取回。下面的数据结构把这两级存储落成代码。


In [ ]:
# 两级记忆存储：主上下文与外部上下文
class MainContext:
    """主上下文：system 指令 + working context + FIFO 消息队列。"""

    def __init__(self, system_text):
        self.system = system_text
        self.working = ""
        self.queue = []

    def tokens(self):
        """估算占用 token 数，这里用字符数近似。"""
        return (len(self.system) + len(self.working)
                + sum(len(m["content"]) for m in self.queue))

    def render(self):
        """按固定顺序拼接成提示文本。"""
        parts = [self.system, "【working context】" + self.working]
        parts += [f"{m['role']}: {m['content']}" for m in self.queue]
        return "\n".join(parts)


class ExternalContext:
    """外部上下文：recall 库（全部历史）与 archival 库（长期档案）。"""

    def __init__(self):
        self.recall = []
        self.archival = {}

    def log(self, role, content):
        """把一条消息永久写入 recall 库。"""
        self.recall.append({"role": role, "content": content})

    def search(self, keyword):
        """在 recall 库里做关键词检索，返回最近命中的至多 3 条。"""
        hits = [m for m in self.recall if keyword in m["content"]]
        return hits[-3:]


main = MainContext("你是助记助手，保持记忆有界。")
ext = ExternalContext()
main.working = "用户：小明，水果：苹果"
main.queue.append({"role": "user", "content": "今天天气不错"})
ext.log("user", "我叫小明，我最爱的水果是苹果")
print("主上下文 token 数:", main.tokens())
print("检索「苹果」命中:", len(ext.search("苹果")), "条")
print("主上下文预览:", main.render()[:44] + "...")


模型通过函数调用来读写自己的记忆。每个函数有一个 schema：名字、参数与自然语言描述。函数执行器解析模型输出里的函数调用，校验参数后执行，把结果（成功信息或错误）回喂给模型。下面实现四个最常用的函数：向 working context 追加、替换 working context、往 archival 写入、检索 recall 库。


In [ ]:
import re


def parse_functions(text):
    """从模型回复里解析 (函数名, 参数) 列表，形如 name("args")。"""
    return re.findall(r"(\w+(?:\.\w+)?)\s*\(\s*\"([^\"]*)\"\s*\)", text)


def execute_function(main, ext, name, args):
    """执行一个记忆函数，更新两级存储，返回结果字符串。"""
    if name == "working_context.append":
        main.working = (main.working + "\n" + args).strip()
        return "已追加：" + args
    if name == "working_context.replace":
        old, new = args.split("->", 1)
        main.working = main.working.replace(old.strip(), new.strip())
        return f"已改写：{old.strip()} -> {new.strip()}"
    if name == "archival.insert":
        key = args.split(":", 1)[0].strip()
        ext.archival[key] = args
        return "已写入档案：" + key
    if name == "recall.search":
        hits = ext.search(args)
        return "；".join(m["content"] for m in hits) if hits else "未找到"
    return "未知函数：" + name


reply = ('Thought: 用户说了新事实。\n'
         'Action: working_context.append("小明喜欢篮球")\n'
         'Action: recall.search("苹果")')
for name, args in parse_functions(reply):
    print("执行:", name, "|", execute_function(main, ext, name, args))
print("working context 现值:", main.working)


把上面的组件拼成一个最小的 MemGPT 式循环。ToyAgent 每轮接收一条消息，写入主上下文与 recall 库；maintain 维护预算，超限时逐出最老消息并生成递归摘要；ask 用关键词检索外部存储，把命中放回主上下文再调用模型。下面先初始化 LLM 客户端，再定义这个 agent。


In [ ]:
import os
import sys

_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)

from llm_client import get_llm

client = get_llm(force_mock=not (os.environ.get("AGENT_LLM_API_KEY")
                                 or os.environ.get("ANTHROPIC_API_KEY")))
print("LLM 客户端:", "mock（占位输出）" if client.is_mock else "real API")


class ToyAgent:
    """最小的 MemGPT 式 agent：两级存储 + 函数 + 预算维护。

    main：MainContext；ext：ExternalContext。每轮 ingest 一条消息，
    然后 maintain 保证主上下文 token 不超过 budget。
    """

    def __init__(self, client, system, budget, warn_ratio=0.7):
        self.client = client
        self.main = MainContext(system)
        self.ext = ExternalContext()
        self.budget = budget
        self.warn = int(budget * warn_ratio)
        self.summary = "（空摘要）"
        self.history_tokens = []

    def ingest(self, role, content):
        """把消息写入主上下文队列与外部 recall 库。"""
        self.main.queue.append({"role": role, "content": content})
        self.ext.log(role, content)

    def maintain(self):
        """维护预算：超限时逐出最老消息并生成递归摘要，直到回到预算内。"""
        events = []
        while self.main.tokens() > self.budget:
            if self.main.tokens() >= self.warn:
                events.append("内存压力警告")
            evicted = self.main.queue.pop(0)
            self.ext.log("system", "evicted: " + evicted["content"])
            self.summary = self._summarize(self.summary, evicted["content"])
            events.append("逐出: " + evicted["content"][:18] + "...")
        self.history_tokens.append(self.main.tokens())
        return events

    def _summarize(self, old, evicted):
        """用 LLM 生成递归摘要，mock 模式退化为确定性拼接。"""
        if self.client.is_mock:
            return old + " ~ " + evicted
        prompt = f"已有摘要：{old}\n新逐出消息：{evicted}\n请合并成新摘要："
        return self.client.chat([{"role": "user", "content": prompt}])

    def ask(self, question, keyword):
        """检索外部记忆放回主上下文，再调用模型回答。"""
        hits = self.ext.search(keyword)
        context = "；".join(m["content"] for m in hits) or "（无相关记忆）"
        self.main.queue.append({"role": "system", "content": "检索记忆: " + context})
        reply = self.client.chat(
            [{"role": "user", "content": question + "\n参考记忆：" + context}])
        self.main.queue.append({"role": "assistant", "content": reply})
        return reply, hits


In [ ]:
# 运行一段脚本化对话：预算很小，让逐出快速发生
system = "你是助记助手。把用户关键事实写进 working context，历史消息存 recall 库。"
agent = ToyAgent(client, system, budget=70)

script = [
    ("user", "我叫小明，我最爱的水果是苹果，住在北京。"),
    ("user", "今天天气不错，适合散步。"),
    ("user", "项目下周五截止，记得提醒我。"),
    ("user", "这周还有什么安排？"),
]
for role, content in script:
    agent.ingest(role, content)
    events = agent.maintain()
    print(f"[{role}] {content[:22]}")
    for e in events:
        print("   ", e)
    print(f"    main tokens = {agent.main.tokens()} / budget {agent.budget}")
print("关键观察：用户事实在轮次间被逐出主上下文，进入外部存储。")


In [ ]:
# 换出再召回：事实已被逐出，ask 从外部存储找回并放回主上下文
before = "苹果" in agent.main.render()
reply, hits = agent.ask("我最爱的水果是什么？", "水果")
after = "苹果" in agent.main.render()
print("提问前「苹果」在主上下文:", before)
print("检索命中:", [h["content"][:24] for h in hits])
print("检索后「苹果」在主上下文:", after)
print("模型回答:", reply, "（mock 模式为占位输出）")
print("关键观察：被逐出的事实经 recall.search 找回，重新进入主上下文。")


In [ ]:
# 可视化：主上下文占用保持在预算内
import matplotlib.pyplot as plt
import numpy as np

turn_ids = np.arange(len(agent.history_tokens))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(turn_ids, agent.history_tokens, marker="o", label="main context")
ax.axhline(agent.budget, color="r", linestyle="--", label="budget")
ax.axhline(agent.warn, color="orange", linestyle="--", label="warning")
ax.set_xlabel("turn")
ax.set_ylabel("tokens")
ax.set_title("MemGPT-style budget maintenance")
ax.legend()
plt.tight_layout()
plt.show()
print(f"每轮主上下文都保持在预算 {agent.budget} 以内。")


## 3. Cartridges：把语料蒸馏进可训练表示

MemGPT 搬运的是文本 token，每一步都要消耗推理。另一种思路完全不搬运文本：把整份语料离线压进一组可训练向量，推理时把这组向量当作前缀键值对拼进模型，用户查询照常解码。这就是前缀调优（prefix-tuning）：往输入前加 p 个虚拟 token，但这次每个语料训练一份，训练成本被反复查询摊薄。Cartridges 把这份虚拟键值对叫作 cartridge。

一个直接的想法是在语料上做 next-token prediction，结果会背诵语料，却答不了问题——背诵不泛化。Cartridges 改用 context-distillation：训练的目标不是复述语料，而是让带 cartridge 的学生复刻"语料放在上下文里"的教师分布。玩具里我们把教师分布写成解析形式：对每个问题，正确答案占 0.85、其余均分；真实系统里它由模型前向得到。下面先用两个小分布手算这个蒸馏目标，再在 torch 里从零实现。


In [ ]:
# 手算：context-distillation 的 KL 目标
import torch


def d_kl(p, q):
    """KL 散度 KL(P || Q) = sum p log(p / q)，p、q 为概率分布。"""
    return (p * torch.log(p / q)).sum().item()


teacher = torch.tensor([0.02, 0.02, 0.90, 0.03, 0.03])    # 语料在上下文
student_before = torch.tensor([0.20, 0.20, 0.20, 0.20, 0.20])  # 初始学生
student_after = torch.tensor([0.02, 0.02, 0.90, 0.03, 0.03])

kl_before = d_kl(teacher, student_before)
kl_after = d_kl(teacher, student_after)
print(f"KL(教师 || 初始学生) = {kl_before:.3f}")
print(f"KL(教师 || 蒸馏后学生) = {kl_after:.3f}")
print("关键观察：蒸馏把学生分布拉到教师附近，KL 降到接近 0。")


In [ ]:
# 从零实现：迷你因果语言模型，支持前缀 KV 与可训练虚拟键值
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(8)   # 限制线程数，避免小张量上的线程抖动


class TinyGPT(nn.Module):
    """迷你因果语言模型，支持在键值序列前拼接 prefix KV。

    prefix_ids 给定时用真实语料作前缀（教师）；否则使用每层的可训练
    虚拟键值向量（学生，即 cartridge）。真实权重全部冻结。
    """

    def __init__(self, vocab, d=48, n_head=4, n_layer=2, p=8):
        super().__init__()
        self.d, self.n_head, self.n_layer, self.p = d, n_head, n_layer, p
        self.embed = nn.Embedding(vocab, d)
        self.vk = nn.ParameterList(
            [nn.Parameter(torch.randn(p, d) * 0.02) for _ in range(n_layer)])
        self.vv = nn.ParameterList(
            [nn.Parameter(torch.randn(p, d) * 0.02) for _ in range(n_layer)])
        self.layers = nn.ModuleList([self._block() for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, vocab, bias=False)

    def _block(self):
        """单层：qkv 投影、输出投影、两层前馈网络、两个 LayerNorm。"""
        return nn.ModuleDict({
            "qkv": nn.Linear(self.d, 3 * self.d),
            "out": nn.Linear(self.d, self.d),
            "ffn": nn.Sequential(nn.Linear(self.d, 2 * self.d),
                                 nn.GELU(),
                                 nn.Linear(2 * self.d, self.d)),
            "ln1": nn.LayerNorm(self.d),
            "ln2": nn.LayerNorm(self.d),
        })

    def _attention(self, q, k, v, n_prefix):
        """因果多头注意力。q 形状 [B,T,d]，k/v 形状 [B,T+n_prefix,d]。"""
        B, T, d = q.shape
        h, dh = self.n_head, d // self.n_head
        q = q.view(B, T, h, dh).transpose(1, 2)
        k = k.view(B, -1, h, dh).transpose(1, 2)
        v = v.view(B, -1, h, dh).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (dh ** 0.5)
        mask = torch.triu(
            torch.ones(T, n_prefix + T, dtype=torch.bool),
            diagonal=n_prefix + 1)
        scores = scores.masked_fill(mask[None, None], float("-inf"))
        w = F.softmax(scores, dim=-1)
        return (w @ v).transpose(1, 2).reshape(B, T, d)

    def forward(self, ids, prefix_ids=None, return_all=False):
        """ids 形状 [B,T]。return_all 时返回 [B,T,vocab]，否则返回
        最后位置的 logits [B,vocab]。"""
        B, T = ids.shape
        x = self.embed(ids)
        for i, blk in enumerate(self.layers):
            xn = blk["ln1"](x)
            qkv = blk["qkv"](xn)
            q, k, v = qkv.chunk(3, dim=-1)
            if prefix_ids is not None:
                pn = blk["ln1"](self.embed(prefix_ids))
                pkv = blk["qkv"](pn)
                pk, pv = pkv[:, :, self.d:2 * self.d], pkv[:, :, 2 * self.d:]
            else:
                pk = self.vk[i].unsqueeze(0).expand(B, -1, -1)
                pv = self.vv[i].unsqueeze(0).expand(B, -1, -1)
            k = torch.cat([pk, k], dim=1)
            v = torch.cat([pv, v], dim=1)
            x = x + blk["out"](self._attention(q, k, v, pk.size(1)))
            x = x + blk["ffn"](blk["ln2"](x))
        logits = self.lm_head(self.ln_f(x))
        return logits if return_all else logits[:, -1, :]


model = TinyGPT(vocab=20)
print("可训练参数数:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("其中虚拟键值参数数:", sum(p.numel() for p in list(model.vk.parameters())
                                  + list(model.vv.parameters())))


In [ ]:
# 玩具语料与问题：语料是一份人物档案字典，问题是档案检索
corpus_tokens = ["name", "ALICE", ".", "fruit", "apple", ".", "city",
                 "Beijing", ".", "name", "BOB", ".", "fruit", "banana",
                 ".", "city", "Shanghai", ".", "name", "EVE", ".", "fruit",
                 "grape", ".", "city", "Paris", "."]
questions = [(("ALICE", "fruit"), "apple"),
             (("BOB", "fruit"), "banana"),
             (("ALICE", "city"), "Beijing"),
             (("BOB", "city"), "Shanghai")]
vocab = sorted(set(corpus_tokens)
               | set(a for _, a in questions)
               | set(t for q, _ in questions for t in q))
stoi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
corpus_ids = torch.tensor([stoi[w] for w in corpus_tokens]).unsqueeze(0)
print("词表:", vocab)
print("语料长度:", corpus_ids.numel(), "token；问题数:", len(questions))
print("问题与答案:", [" ".join(q) + " -> " + a for q, a in questions])


In [ ]:
# 冻结真实权重 + 教师目标：正确答案占 0.85，其余均分
def freeze_except_virtual(model):
    """冻结全部真实权重，只保留每层的虚拟键值可训练。"""
    for p in model.parameters():
        p.requires_grad_(False)
    for name, p in model.named_parameters():
        if "vk" in name or "vv" in name:
            p.requires_grad_(True)


def make_teacher(q, ans):
    """按问题与答案构造教师分布，代表"语料在上下文"时的答题分布。"""
    p = torch.full([V], 0.15 / V)
    p[stoi[ans]] = 0.85
    return p


teacher_targets = {tuple(q): make_teacher(q, ans) for q, ans in questions}
q0, a0 = questions[0]
top = torch.topk(teacher_targets[tuple(q0)], 3)
print(f"教师分布（问题 {' '.join(q0)}，答案 {a0}）：")
for val, idx in zip(top.values, top.indices):
    print(f"  {vocab[idx]:8s} {val.item():.2f}")


In [ ]:
# 目标一：NTP cartridge——在语料上做 next-token prediction（背诵）
torch.manual_seed(9)
model_ntp = TinyGPT(V)
freeze_except_virtual(model_ntp)
opt = torch.optim.Adam([p for p in model_ntp.parameters() if p.requires_grad],
                       lr=0.05)
losses = []
for step in range(400):
    opt.zero_grad()
    logits = model_ntp(corpus_ids[:, :-1], return_all=True)
    loss = F.cross_entropy(logits.reshape(-1, V),
                           corpus_ids[:, 1:].reshape(-1))
    loss.backward()
    opt.step()
    losses.append(loss.item())
print("NTP cartridge 末轮 loss:", round(losses[-1], 3),
      "（目标是把语料续写压进表示，不是答题）")


In [ ]:
# 目标二：context-distillation——学生复刻教师分布
torch.manual_seed(9)
model_distill = TinyGPT(V)
freeze_except_virtual(model_distill)
opt = torch.optim.Adam(
    [p for p in model_distill.parameters() if p.requires_grad], lr=0.05)
kls = []
for step in range(500):
    opt.zero_grad()
    loss = 0.0
    for q, ans in questions:
        qids = torch.tensor([stoi[w] for w in q]).unsqueeze(0)
        s_log = torch.log_softmax(model_distill(qids), dim=-1)
        loss = loss + F.kl_div(s_log, teacher_targets[tuple(q)],
                               reduction="sum")
    loss = loss / len(questions)
    loss.backward()
    opt.step()
    kls.append(loss.item())
print("context-distillation 末轮 KL:", round(max(0.0, kls[-1]), 3))


In [ ]:
# 对比评估：NTP cartridge 与蒸馏 cartridge 面对问题时的表现
def cartridge_prob(model, q, ans):
    """cartridge 对问题输出正确答案的概率。"""
    qids = torch.tensor([stoi[w] for w in q]).unsqueeze(0)
    with torch.no_grad():
        logits = model(qids)
    return torch.softmax(logits, dim=-1)[0, stoi[ans]].item()


def kl_to_teacher(model, q):
    """cartridge 分布与教师分布的 KL(教师 || cartridge)。"""
    qids = torch.tensor([stoi[w] for w in q]).unsqueeze(0)
    with torch.no_grad():
        s_log = torch.log_softmax(model(qids), dim=-1)
    p = teacher_targets[tuple(q)]
    return max(0.0, (p * (p.log() - s_log)).sum().item())


import matplotlib.pyplot as plt
import numpy as np

labels = [" ".join(q) for q, _ in questions]
p_ntps = [cartridge_prob(model_ntp, q, ans) for q, ans in questions]
p_dls = [cartridge_prob(model_distill, q, ans) for q, ans in questions]
print("问题 -> 答案 | NTP P | distill P | NTP KL | distill KL")
for i, (q, ans) in enumerate(questions):
    k_ntp = kl_to_teacher(model_ntp, q)
    k_dl = kl_to_teacher(model_distill, q)
    print(f"{' '.join(q):16s} -> {ans:8s} | {p_ntps[i]:5.2f} | "
          f"{p_dls[i]:9.2f} | {k_ntp:6.2f} | {k_dl:7.2f}")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(labels))
ax.bar(x - 0.2, p_ntps, 0.4, label="NTP cartridge")
ax.bar(x + 0.2, p_dls, 0.4, label="distill cartridge")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel("P(correct answer)")
ax.set_title("Question answering: NTP vs distillation")
ax.legend()
plt.tight_layout()
plt.show()

print(f"平均正确概率：NTP {np.mean(p_ntps):.2f}，蒸馏 {np.mean(p_dls):.2f}")
print(f"平均 KL(教师||学生)：NTP "
      f"{np.mean([kl_to_teacher(model_ntp, q) for q, _ in questions]):.2f}，"
      f"蒸馏 {np.mean([kl_to_teacher(model_distill, q) for q, _ in questions]):.2f}")
print("关键观察：蒸馏对每个问题都给出高概率正确回答，NTP 只对背到的模式有效。")


## 4. CacheBlend：跨 chunk 复用 KV 缓存

RAG 把输入拼成多个文本 chunk，prefill（对整段输入算 KV cache）很慢，直接决定了首 token 延迟。已有两类复用方案各有短板：prefix caching 只复用前缀 chunk 的 KV，其他 chunk 照常 prefill；full KV reuse 复用所有 chunk，却忽略了 chunk 之间的跨 chunk 注意力，质量受损。CacheBlend 提出选择性重算：按层只重算一小部分 token 的 KV，其余沿用缓存，同时拿到复用速度与全量重算质量。

直觉来自注意力稀疏性：只有少数"跨界"token 的 KV 在单独 prefill 与整段 prefill 之间偏离最大，重算它们就足够恢复跨 chunk 注意力。下面先手算"token 级相似度加权融合"的注意力分数，再看三种方案的注意力矩阵。


In [ ]:
# 手算：token 级相似度加权融合的注意力分数
import numpy as np

# 4 个上下文 token：chunk A = [a1, a2]，chunk B = [b1, b2]
# 全量重算：整段一起 prefill 得到真实 key
K_full = np.array([
    [1.0, 0.0, 0.0, 0.0],   # a1
    [0.0, 1.0, 0.0, 0.0],   # a2
    [0.0, 0.0, 1.0, 0.0],   # b1
    [0.0, 0.0, 0.0, 1.0],   # b2
])
# 缓存 KV：两个 chunk 各自独立 prefill，跨界 token 偏离真实值
K_cache = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.5, 0.8, 0.2, 0.1],   # a2 受 chunk B 影响而偏离
    [0.1, 0.2, 0.9, 0.2],   # b1 受 chunk A 影响而偏离
    [0.0, 0.0, 0.0, 1.0],
])
q = np.array([0.7, 0.3, 0.5, 0.2])   # query token，实时计算


def attn_scores(q, K):
    """token 级注意力分数：query 与每个 key 的点积。"""
    return q @ K.T


s_full = attn_scores(q, K_full)
s_cache = attn_scores(q, K_cache)
dev = np.linalg.norm(K_cache - K_full, axis=1)
print("全量重算 s_full:", np.round(s_full, 2))
print("缓存复用 s_cache:", np.round(s_cache, 2))
print("逐 token 偏差   :", np.round(np.abs(s_cache - s_full), 2))
print("KV deviation    :", np.round(dev, 2))

r = 0.5
k = int(np.ceil(r * len(K_full)))
hkvd = np.argsort(dev)[::-1][:k]
print("HKVD 重算 token :", sorted(hkvd.tolist()), "(前", int(r * 100), "%)")
K_cb = K_cache.copy()
K_cb[hkvd] = K_full[hkvd]
s_cb = attn_scores(q, K_cb)
print("CacheBlend s_cb :", np.round(s_cb, 2))
print("融合后偏差      :", np.round(np.abs(s_cb - s_full), 2))
print("关键观察：重算 2 个跨界 token，注意力分数回到全量重算。")


In [ ]:
# 三种方案的注意力矩阵对比
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
d = 8
n_a, n_b = 3, 3
T = n_a + n_b
Q = rng.normal(size=(T, d))
K_full = rng.normal(size=(T, d))
V_full = rng.normal(size=(T, d))
# 跨界 token 离 chunk 边界越近，缓存偏离越大
dist = np.array([0, 1, 2, 2, 1, 0])
K_cache = K_full + dist[:, None] * rng.normal(size=(T, d))


def attn_matrix(Q, K, V):
    """单头注意力矩阵，softmax(Q K^T / sqrt(d))。"""
    s = Q @ K.T / (d ** 0.5)
    e = np.exp(s - s.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


A_full = attn_matrix(Q, K_full, V_full)
# 全量复用：两个 chunk 各自独立算注意力，跨 chunk 块被抹零
A_reuse = np.zeros_like(A_full)
A_reuse[:n_a, :n_a] = attn_matrix(Q[:n_a], K_full[:n_a], V_full[:n_a])
A_reuse[n_a:, n_a:] = attn_matrix(Q[n_a:], K_full[n_a:], V_full[n_a:])
# CacheBlend：重算 KV deviation 最高的 token
dev = np.linalg.norm(K_cache - K_full, axis=1)
k = int(np.ceil(0.5 * T))
hkvd = np.argsort(dev)[::-1][:k]
K_cb = K_cache.copy()
K_cb[hkvd] = K_full[hkvd]
A_cb = attn_matrix(Q, K_cb, V_full)


def fro_norm(A, B):
    """两个注意力矩阵的 F 范数差。"""
    return np.linalg.norm(A - B)


print("注意力偏差 ||A_reuse - A_full||:", round(fro_norm(A_reuse, A_full), 3))
print("注意力偏差 ||A_cb   - A_full||:", round(fro_norm(A_cb, A_full), 3))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
titles = ["full recompute", "full KV reuse", "CacheBlend"]
for ax, A, title in zip(axes, [A_full, A_reuse, A_cb], titles):
    ax.imshow(A, cmap="YlOrRd", vmin=0, vmax=0.6)
    ax.set_title(title)
    ax.set_xlabel("key token")
    ax.set_ylabel("query token")
plt.tight_layout()
plt.show()
print("热图颜色越深注意力越高；full reuse 的跨 chunk 块被抹零。")


In [ ]:
# KV deviation 的稀疏性：少数 token 贡献大部分偏离
import numpy as np
import matplotlib.pyplot as plt

dev_sorted = np.sort(dev)[::-1]
cdf = np.cumsum(dev_sorted) / dev_sorted.sum()
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(np.arange(1, T + 1), cdf, marker="o")
ax.axhline(0.9, color="gray", linestyle="--", label="90% deviation")
ax.set_xlabel("top-k tokens")
ax.set_ylabel("cumulative deviation fraction")
ax.set_title("KV deviation is concentrated")
ax.legend()
plt.tight_layout()
plt.show()
top_k = int(np.flatnonzero(cdf >= 0.9)[0]) + 1
print(f"前 {top_k} 个 token 贡献了 90% 的 deviation")

# 同样重算 k 个 token：HKVD 与随便选（低偏差 token）谁恢复得更好
rand = np.argsort(dev)[:k]      # 故意选偏离最小的 k 个


def recovered(K):
    """用给定 key 矩阵重算注意力相对全量的偏差。"""
    A = attn_matrix(Q, K, V_full)
    return np.linalg.norm(A - A_full)


K_hkvd = K_cache.copy()
K_hkvd[hkvd] = K_full[hkvd]
K_rand = K_cache.copy()
K_rand[rand] = K_full[rand]
print("HKVD 重算注意力偏差:", round(recovered(K_hkvd), 3))
print("低偏差 token 重算偏差:", round(recovered(K_rand), 3))
print("关键观察：同样的重算预算，HKVD 恢复得接近全量，随意选则无效。")


In [ ]:
# 渐进筛选：相邻层 KV deviation 高度相关，不必每层全量算
import numpy as np
from scipy.stats import spearmanr


def top_r_percent(dev, r):
    """返回 KV deviation 中位于前 r 比例的 token 索引。"""
    k = max(1, int(np.ceil(r * len(dev))))
    return np.argsort(dev)[::-1][:k]


rng = np.random.default_rng(7)
base_dev = rng.uniform(0.1, 1.0, size=12)
devs = [np.clip(base_dev + rng.normal(scale=0.05 * (l + 1), size=12),
                0, None) for l in range(4)]

ratios = [0.30, 0.20, 0.15, 0.10]
cand = None
for l, (dev, r) in enumerate(zip(devs, ratios)):
    if cand is None:
        cand = top_r_percent(dev, r)
    else:
        cand = cand[top_r_percent(dev[cand], r / ratios[l - 1])]
    print(f"layer {l}: 候选 {len(cand)} 个")

for l in range(len(devs) - 1):
    rho, _ = spearmanr(devs[l], devs[l + 1])
    print(f"layer {l} vs {l + 1} Spearman 秩相关 = {rho:.3f}")
print("关键观察：相邻层 deviation 高度相关，可逐层在候选内筛选。")


## 5. 记忆系统的工程实践

把三篇论文连起来看，记忆系统的工作分成三层：MemGPT 决定记忆放在哪一级存储，Cartridges 决定记忆长成什么表示，CacheBlend 让同一段记忆被反复使用时不必重新付出 prefill 的成本。

CacheBlend 的系统工程把选择性重算与从慢速存储加载 KV 排成流水线：重算第 i 层时，后台加载第 i+1 层。只要重算时间不超过加载时间，重算就是免费的，KV 缓存因此可以放到更慢、更便宜的设备上而不增加首 token 延迟。下面用一个延迟模型复现这个决策。


In [ ]:
# 延迟模型：重算与 KV 加载的流水线
import numpy as np
import matplotlib.pyplot as plt

# 每层 KV 缓存大小（Llama-7B 量级，2048 token 上下文，fp16）
kv_bytes = 2048 * 2 * 4096 * 2          # token × (K+V) × dim × 2 字节
throughput = {"HBM": 4e12, "NVMe SSD": 2e9, "SATA SSD": 5.5e8, "HDD": 1.5e8}


def T_load(device):
    """从 device 加载一层 KV 的耗时（毫秒）。"""
    return kv_bytes / throughput[device] * 1000


def T_recompute(r, prefill_layer_ms=20):
    """重算 r 比例 token 的单层耗时（毫秒）。"""
    return r * prefill_layer_ms


print("单层 KV 缓存:", round(kv_bytes / 1e6, 1), "MB")
print("重算 15% token 单层:", f"{T_recompute(0.15):.1f} ms")
for dev in throughput:
    t = T_load(dev)
    hidden = "重算被加载延迟隐藏" if t >= T_recompute(0.15) else "重算成为瓶颈"
    print(f"{dev:8s} 加载单层 {t:7.1f} ms  | {hidden}")

# 流水线延迟 = max(重算, 加载)
rs = np.linspace(0, 1, 100)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rs, np.maximum(T_recompute(rs), T_load("NVMe SSD")), label="NVMe SSD")
ax.plot(rs, np.maximum(T_recompute(rs), T_load("HBM")), label="HBM")
ax.axvline(0.15, color="gray", linestyle="--", label="quality floor 15%")
ax.set_xlabel("recompute ratio r")
ax.set_ylabel("per-layer latency (ms)")
ax.set_title("Pipelined recompute vs KV load")
ax.legend()
plt.tight_layout()
plt.show()
print("NVMe 曲线在 r < 0.84 时保持平直：加载延迟盖住了重算延迟。")


## 小结

这一讲完成了三层记忆的实现：

- [ ] 记忆为什么重要：上下文窗口是硬上限，逐出策略决定让位顺序，FIFO / 重要性 / LRU 各有权衡
- [ ] MemGPT：main context 与 external context 两级存储，函数机制让模型自己读写记忆，内存压力触发逐出与递归摘要
- [ ] 逐出与预算：70% 警告线、100% 冲水线，逐出后主上下文回到预算内，被逐事实可经 recall 召回
- [ ] Cartridges：把语料蒸馏进可训练键值向量，context-distillation 让学生复刻"语料在上下文"的输出分布
- [ ] CacheBlend：跨 chunk 复用预计算 KV，选择性重算高偏差 token 即可恢复跨 chunk 注意力
- [ ] 工程实践：重算与 KV 加载流水线重叠，慢速存储也能隐藏重算延迟

下一条主线是评测：Agent 的记忆与工具做得对不对，需要一个能长期运行、能区分真进步与假表现的评测框架。


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。

三道小题都基于本讲写过的代码，先在草稿上自己补全一遍，再运行对照。


**作业 1：MemGPT 队列的逐出与递归摘要**

在下面的函数里补全两处：`should_flush` 判断主上下文是否到达冲水线；`make_summary` 把旧摘要与被逐消息合成一条递归摘要。断言验证冲水线判断与逐出后的预算。

小提示：逐出的是队列最老的一条消息；递归摘要要把旧摘要与被逐消息都考虑进去。


In [ ]:
# 填空 1：判断是否到达冲水线
def should_flush(tokens, budget):
    """主上下文 token 数是否达到冲水线。"""
    return tokens >= budget


# 填空 2：递归摘要 = 旧摘要 + 被逐消息
def make_summary(client, old, evicted):
    """把旧摘要与被逐消息合并成一条递归摘要。"""
    if client.is_mock:
        return old + " ~ " + evicted
    prompt = f"旧摘要：{old}\n被逐消息：{evicted}\n合并成新摘要："
    return client.chat([{"role": "user", "content": prompt}])


budget = 60
msgs = ["m" * 26, "n" * 26, "p" * 26]          # 每条 26 字符
total = sum(len(m) for m in msgs)
assert should_flush(total, budget) is True, "78 >= 60 应冲水"
left = total - len(msgs[0])
assert should_flush(left, budget) is False, "逐出一条后应回到预算内"
summary = make_summary(client, "（空）", msgs[0])
assert len(summary) > 0, "递归摘要不能为空"
print("作业 1 通过：能判断冲水线，逐出后回到预算内，递归摘要已生成。")


**作业 2：HKVD 选择与层间相关**

补全 `top_r_percent` 选出 KV deviation 最高的前 r 比例 token；补全 `layer_correlation` 用 Spearman 秩相关度量相邻两层 deviation 的一致性。断言验证选出的集合与相关性阈值。

小提示：`np.argsort(dev)[::-1]` 得到降序下标；`spearmanr` 返回 (rho, pvalue) 两个值。


In [ ]:
import numpy as np
from scipy.stats import spearmanr


def top_r_percent(dev, r):
    """返回 KV deviation 中位于前 r 比例的 token 索引。"""
    k = max(1, int(np.ceil(r * len(dev))))
    return np.argsort(dev)[::-1][:k]


def layer_correlation(dev_a, dev_b):
    """计算两层 KV deviation 的 Spearman 秩相关。"""
    rho, _ = spearmanr(dev_a, dev_b)
    return rho


dev_l0 = np.array([0.1, 0.9, 0.2, 0.8, 0.15, 0.7])
dev_l1 = np.array([0.12, 0.85, 0.18, 0.82, 0.13, 0.72])
assert set(top_r_percent(dev_l0, 0.5)) == {1, 3, 5}, "应选出 deviation 最大的 3 个"
assert layer_correlation(dev_l0, dev_l1) > 0.9, "相邻层 deviation 应高度相关"
print("作业 2 通过：HKVD 选出前 50% token，层间 Spearman 相关高于 0.9。")


**作业 3：context-distillation 的 KL 目标**

补全 `distillation_loss`：把教师 logits 与学生对数 logits 转成分布后求 KL(教师 || 学生)。断言验证对齐教师分布的损失低于均匀学生。

小提示：教师用 softmax 转成概率，学生用 log_softmax 求对数概率；KL = sum p (log p - log q)。


In [ ]:
import torch


def distillation_loss(teacher_logits, student_logits):
    """KL(教师 || 学生)，教师来自语料在上下文，学生来自 cartridge。"""
    p = torch.softmax(teacher_logits, dim=-1)
    log_q = torch.log_softmax(student_logits, dim=-1)
    return (p * (p.log() - log_q)).sum(dim=-1).mean()


teacher_logits = torch.tensor([[0.1, 0.1, 2.0, 0.2, 0.1]])
student_uniform = torch.tensor([[0.3, 0.3, 0.3, 0.3, 0.3]])
student_aligned = torch.tensor([[0.1, 0.1, 2.0, 0.2, 0.1]])
loss_bad = distillation_loss(teacher_logits, student_uniform)
loss_good = distillation_loss(teacher_logits, student_aligned)
assert loss_good.item() < loss_bad.item(), "对齐分布应让 KL 变小"
print("作业 3 通过：KL 目标让 cartridge 对齐教师的答题分布。")


## 参考资料

- Packer et al., [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560), 2023 — OS 式分层记忆：main/external context 两级存储与模型自导向的函数读写
- Berglund et al., [Cartridges: Lightweight and general-purpose long context representations via self-study](https://arxiv.org/abs/2506.06266), 2025 — 把语料离线蒸馏进可训练 KV 表示，self-study 合成数据与 context-distillation 目标
- Huang et al., [CacheBlend: Fast LLM Serving for RAG with Cached Knowledge Fusion](https://arxiv.org/abs/2405.16444), 2024 — 跨 chunk 复用预计算 KV，选择性重算高偏差 token 恢复跨 chunk 注意力
- Liu et al., [Lost in the Middle: How Language Models Use Long Contexts](https://arxiv.org/abs/2307.03172), 2023 — 长上下文模型中间信息丢失的实证，MemGPT 的动机来源
- Li & Liang, [Prefix-Tuning: Optimizing Continuous Prompts for Generation](https://arxiv.org/abs/2101.00190), 2021 — 可训练前缀向量的原始工作，Cartridge 参数化的来源
- Chen et al., [PromptCache: Modular Attention Reuse for Low-latency Inference](https://arxiv.org/abs/2311.04934), 2023 — 全量 KV 复用基线，忽略跨 chunk 注意力的那一种
- [Letta](https://github.com/letta-ai/letta) — MemGPT 的开源工程化实现，长期记忆 agent 框架
- CS329A 课程主页（https://cs329a.stanford.edu/）— Autumn 2025 课程大纲，本讲在"给 Agent 加记忆"单元
